In [1]:
import urllib.request

url = 'https://github.com/DataTalksClub/machine-learning-zoomcamp/releases/download/chapter7-model/xception_v4_large_08_0.894.h5'
urllib.request.urlretrieve(url, 'clothing-model.h5')

('clothing-model.h5', <http.client.HTTPMessage at 0x2590d10c9d0>)

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

tf.__version__


'2.21.0'

In [3]:
urllib.request.urlretrieve('http://bit.ly/mlbookcamp-pants', 'pants.jpg')

('pants.jpg', <http.client.HTTPMessage at 0x25928d8b020>)

In [4]:
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.applications.xception import Xception

In [19]:
base_model = Xception(weights='imagenet',include_top=False,input_shape=(150,150,3))

In [20]:
base_model.trainable= False
inputs = keras.Input(shape=(150, 150, 3))
base = base_model(inputs, training=False)
pooling = keras.layers.GlobalAveragePooling2D()(base)
outputs = keras.layers.Dense(10)(pooling)
model = keras.Model(inputs, outputs)

In [21]:
model.load_weights('clothing-model.h5',by_name=True, skip_mismatch=True)

In [22]:
img = load_img('pants.jpg', target_size=(150, 150))

x = np.array(img)
X = np.array([x])

X = preprocess_input(X)

In [23]:
preds = model.predict(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step


In [24]:
preds

array([[ 0.18842797, -1.1006212 , -0.31693298,  0.3098386 , -1.3379817 ,
        -0.7250411 ,  0.5023535 , -0.28525555,  1.0210906 , -0.55201095]],
      dtype=float32)

In [25]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

In [26]:
dict(zip(classes, preds[0]))

{'dress': np.float32(0.18842797),
 'hat': np.float32(-1.1006212),
 'longsleeve': np.float32(-0.31693298),
 'outwear': np.float32(0.3098386),
 'pants': np.float32(-1.3379817),
 'shirt': np.float32(-0.7250411),
 'shoes': np.float32(0.5023535),
 'shorts': np.float32(-0.28525555),
 'skirt': np.float32(1.0210906),
 't-shirt': np.float32(-0.55201095)}

In [12]:
!pip install keras-image-helper


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

tflite_model = converter.convert()

with open('clothing-model.tflite', 'wb') as f_out:
    f_out.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmpabnmcy2c\assets


INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmpabnmcy2c\assets


Saved artifact at 'C:\Users\user\AppData\Local\Temp\tmpabnmcy2c'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150, 150, 3), dtype=tf.float32, name='keras_tensor_264')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2582422680016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422681360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422681168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422677712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422678288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582386275920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422677328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422677520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422683472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2582422682320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2

In [14]:
!ls -lh

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [28]:
import tensorflow.lite as tflite

In [33]:
intepreter = tflite.Interpreter(model_path='clothing-model.tflite')
intepreter.allocate_tensors()

input_index = intepreter.get_input_details()[0]['index']
output_index = intepreter.get_output_details()[0]['index']

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [34]:
intepreter.set_tensor(input_index,X)
intepreter.invoke()
preds = intepreter.get_tensor(output_index)

In [35]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

dict(zip(classes, preds[0]))

{'dress': np.float32(0.18842834),
 'hat': np.float32(-1.100621),
 'longsleeve': np.float32(-0.31693187),
 'outwear': np.float32(0.30983797),
 'pants': np.float32(-1.3379815),
 'shirt': np.float32(-0.7250415),
 'shoes': np.float32(0.50235295),
 'shorts': np.float32(-0.2852564),
 'skirt': np.float32(1.0210891),
 't-shirt': np.float32(-0.5520103)}

**Removing TF dependency**

In [36]:
from PIL import Image

In [37]:
with Image.open('pants.jpg') as img:
    img = img.resize((150,150),Image.NEAREST)

In [38]:
def preprocess_input(x):
    x /= 127.5
    x -= 1.
    return x 

In [39]:
x = np.array(img,dtype='float32')
X = np.array([x])

X = preprocess_input(X)

In [40]:
intepreter.set_tensor(input_index,X)
intepreter.invoke()
preds = intepreter.get_tensor(output_index)

In [41]:
classes = [
    'dress',
    'hat',
    'longsleeve',
    'outwear',
    'pants',
    'shirt',
    'shoes',
    'shorts',
    'skirt',
    't-shirt'
]

dict(zip(classes, preds[0]))

{'dress': np.float32(0.18842834),
 'hat': np.float32(-1.100621),
 'longsleeve': np.float32(-0.31693187),
 'outwear': np.float32(0.30983797),
 'pants': np.float32(-1.3379815),
 'shirt': np.float32(-0.7250415),
 'shoes': np.float32(0.50235295),
 'shorts': np.float32(-0.2852564),
 'skirt': np.float32(1.0210891),
 't-shirt': np.float32(-0.5520103)}